# Planners

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/08_planners.ipynb)

Every planner in minilink reads a `PlanningProblem` and returns a `PlanningSolution` from `solve()`. This tutorial presents the public API of `Planner` (`TrajectoryOptimizationPlanner`, `LQRPlanner`, `DynamicProgrammingPlanner`), `PlanningSolution`, `Comparison`, the pure compiled planner face `planner.compile()`, and `ModelPredictiveController`.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. `Planner` and `PlanningSolution`: public API

### Common `Planner` methods

| Method / Property | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `planner.problem` | — | `PlanningProblem` | The declarative planning problem |
| `planner.last_solution` | — | `PlanningSolution \| None` | The very object the latest `solve()` returned, not a copy: the default of the planner's own plots and `get_controller()`; a script keeps the solution `solve()` returns |
| `planner.solve(...)` | `*, params=None, evaluate=False, n_trials=50` (`guess=None` on trajopt and MPPI; the RL planners take their training length first, the deep RL planner no `params=`) | `PlanningSolution` | Solve the problem and store the result in `last_solution` (`evaluate=True` also scores the resulting policy; `params`, the complete tree of `planner.params` edited, `p = copy.deepcopy(planner.params)`, solves there instead) |
| `planner.params` | — | `dict` | The planner's numbers: the problem's leaves by role, `"method"` the method's own (`gamma`, `learning_rate`, `goal_bias`, ...), `"seed"` |
| `planner.get_controller()` | — | `System` | Return `last_solution.policy` |
| `planner.plot_solution(...)` | `*, signals=("x", "u"), backend="matplotlib"` | `PlotResult` | Plot `last_solution`'s trajectory |
| `planner.plot_control_law(**kwargs)` | `x_axis=..., y_axis=..., u_axis=..., bounds=...` | `(Figure, Axes)` | Plot feedback control law over the state box |
| `planner.plot_cost_to_go(**kwargs)` | `axes=(0, 1), anchor=None, bounds=None, jmax=None` | `(Figure, Axes)` | Plot cost-to-go heatmap $J^\star(x)$ |
| `trajopt.compile(backend=None)` | `backend: "numpy" \| "jax" \| None = None` | `CompiledTrajectoryPlanner` | Compile the planner into a stateless functional face `pv` |

### `PlanningSolution` fields, properties, and methods

| Field / Property / Method | Type / Signature | Meaning |
| --- | --- | --- |
| `sol.problem` | `PlanningProblem` | Problem that was solved |
| `sol.policy` | `System` | Control law: open-loop `TrajectorySource` ($u = \pi(t)$, wire with `policy >> plant`) or feedback `System` ($u = \pi(x)$, wire with `policy @ plant`) |
| `sol.record` | Solver record dataclass (`TrajectoryOptimizationRecord`, `RiccatiRecord`, `ValueIterationRecord`, ...) | The solve's whole account: convergence, iterations, constraint checks, and the method's own output (trajopt: `optimization_result`, `history`, `solve_time_s`, `step_time_s`; value iteration and tabular: `tables`, the `J` and `pi` tables) |
| `sol.trajectory` | `Trajectory \| None` | Planned trajectory (trajopt/search) or nominal rollout when `solve(evaluate=True)` is used |
| `sol.evaluation` | `Evaluation \| None` | Closed-loop `Evaluation` when `solve(evaluate=True)` is used |
| `sol.cost_to_go` | `Callable[[x], float] \| None` | Optimal value function $J^\star(x)$ (for DP, LQR, and critic-based learners) |
| `sol.success`, `sol.open_loop`, `sol.method` | `bool`, `bool`, `str` | Feasibility/convergence flag, whether `policy` is open-loop ($m=0$), and human-readable method name |
| `sol.plot_trajectory(...)` | `*, signals=("x", "u"), backend="matplotlib", show=True` | Plot `sol.trajectory` titled as plan or nominal rollout |
| `sol.plot_control_law(**kwargs)` | keyword arguments forwarded to `policy.plot_control_law` | Plot 1D/2D slice of a feedback policy over `problem.X` |
| `sol.plot_cost_to_go(**kwargs)` | `*, axes=(0, 1), anchor=None, bounds=None, jmax=None, show=True` | Plot 2D heatmap of `sol.cost_to_go` over `problem.X` |
| `sol.plot_cost(...)` | `*, backend="matplotlib", show=True` | Plot running cost density and cumulative cost along `sol.trajectory` |

In [ ]:
import numpy as np

from minilink import Pendulum, PlanningProblem, QuadraticCost
from minilink.planning import TrajectoryOptimizationPlanner

plant = Pendulum()
UP = np.array([np.pi, 0.0])
cost = QuadraticCost.from_system(plant, x_bar=UP, Q=np.eye(2), R=0.1 * np.eye(1))
problem = PlanningProblem(
    plant, x0=np.zeros(2), cost=cost, U=plant.input_box, Xf=UP, horizon=5.0
)

trajopt = TrajectoryOptimizationPlanner(
    problem, n_steps=20, transcription="direct_collocation", backend="jax"
)
plan = trajopt.solve(evaluate=True)
print(plan)

In [ ]:
(
    plan.success,
    plan.open_loop,
    plan.method,
    type(plan.policy).__name__,
    plan.trajectory.x.shape,
    plan.evaluation.objective,
)

In [ ]:
plan.plot_trajectory()

## 2. Comparing planners (`LQRPlanner`, `DynamicProgrammingPlanner`) and `Comparison` overlays

Policy planners (`LQRPlanner`, `DynamicProgrammingPlanner`) solve for a feedback law $u = \pi(x)$ and a value function `sol.cost_to_go(x) -> float`. Passing `evaluate=True` to `solve()` also rolls the law out from `problem.x0` to populate `sol.trajectory` and `sol.evaluation`.

| `Comparison` Method / Property | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `compare(Name=sol, ...)` | keyword `PlanningSolution`s | `Comparison` | Align solutions side by side without re-simulating |
| `compare(problem, Name=entry, ..., dt=...)` | `problem: PlanningProblem`, keyword solutions/controllers, `*, dt: float, n_trials=None, key=0` | `Comparison` | Evaluate all entries on paired draws at period `dt` |
| `comp[name]`, `comp.solutions`, `comp.evaluations` | `name: str` | `PlanningSolution`, `dict`, `dict \| None` | Access individual solutions and their `Evaluation` records |
| `comp.difference(a, b)` | `a: str, b: str` | `ndarray (n_trials,)` | Paired per-trial score difference $J_a - J_b$ |
| `comp.plot_control_law(**kwargs)` | `*, show=True, **kwargs` | `(Figure, list[Axes])` | Side-by-side feedback law panels on a shared color scale |
| `comp.plot_cost_to_go(**kwargs)` | `*, jmax=None, show=True, **kwargs` | `(Figure, list[Axes])` | Side-by-side cost-to-go heatmaps on a shared color scale |
| `comp.plot_trajectory(**kwargs)` | `*, signals=("x", "u"), show=True` | `(Figure, list[Axes])` | Overlay planned/nominal trajectories on shared axes |

In [ ]:
from minilink import DynamicProgrammingPlanner, LQRPlanner, compare

regulation = problem.replace(horizon=None, Xf=None, discount_rate=0.2)

lqr_solution = LQRPlanner(regulation).solve(evaluate=True)
value_iteration = DynamicProgrammingPlanner(
    regulation, x_grid=(81, 81), u_grid=(11,), dt=0.05, tol=0.1
)
vi_solution = value_iteration.solve(evaluate=True)

comparison = compare(TO=plan, LQR=lqr_solution, VI=vi_solution)
print(comparison)

In [ ]:
# Querying the learned value function J*(x) at the bottom [0, 0] and upright [pi, 0]
vi_solution.cost_to_go(np.zeros(2)), vi_solution.cost_to_go(UP), lqr_solution.cost_to_go(UP)

In [ ]:
fb_comparison = compare(LQR=lqr_solution, VI=vi_solution)

In [ ]:
fb_comparison.plot_control_law()

In [ ]:
fb_comparison.plot_cost_to_go()

Because `vi_solution.policy` is a `System`, closing the feedback loop on `plant` uses `@`:

In [ ]:
vi_loop = vi_solution.policy @ plant
plant.x0 = np.zeros(2)
vi_traj = vi_loop.compute_trajectory(tf=8.0, verbose=False)

In [ ]:
vi_loop.plot_trajectory(vi_traj)

## 3. `CompiledTrajectoryPlanner`: the pure functional face `pv = planner.compile()`

`pv = planner.compile()` lowers a trajectory-optimization planner into pure functions of a packed plan vector `z` (of length `pv.n_z`, with the plan's time origin `t0` stored in `z[-1]`) and `params`:

| `CompiledTrajectoryPlanner` API | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `pv.n_z`, `pv.knot_spacing`, `pv.predicts` | — | `int`, `float`, `bool` | Size of `z`, time step between knots, and whether `z` carries state knots |
| `pv.params` | — | `dict` | Default parameter tree (`x0`, `t0`, `plant`, `cost`, `method`, `seed`) |
| `pv.initial(params=None)` | `params: dict \| None = None` | `array (n_z,)` | Default initial guess vector `z0` at `params` |
| `pv.solve(params=None, z0=None)` | `params: dict \| None = None`, `z0: array (n_z,) \| None = None` | `(z: array (n_z,), status: dict)` | Solve from `z0` at `params`; `status` has keys `{"success", "iterations", "cost"}` |
| `pv.command(z, x, t, params=None)` | `z: (n_z,)`, `x: (n,)`, `t: float`, `params=None` | `array (m,)` | Control input $u(t)$ interpolated from the plan `z` at time `t` |
| `pv.shift(z, t, x0=None)` | `z: (n_z,)`, `t: float`, `x0: (n,) \| None = None` | `array (n_z,)` | Rebase plan `z` to new start time `t` (and pin first state knot to `x0` when carried) |
| `pv.predict_start(x, u, t, delta, params=None)` | `x: (n,)`, `u: (m,)`, `t: float`, `delta: float`, `params=None` | `array (n,)` | Predict state $\hat x(t + \Delta)$ under held control `u` for delay compensation |
| `pv.prediction(z, t, params=None)` | `z: (n_z,)`, `t: float`, `params=None` | `(x: array (n,), u: array (m,))` | Interpolated state and input at time `t` (collocation / multiple shooting) |
| `pv.planned_trajectory(z, params=None)` | `z: (n_z,)`, `params=None` | `Trajectory` | Knot schedule as a `Trajectory` (collocation / multiple shooting) |
| `pv.solution(z, params=None, status=None)` | `z: (n_z,)`, `params=None`, `status=None` | `PlanningSolution` | Reconstruct a full host-side `PlanningSolution` from `z` |
| `pv.pack(guess, params=None)` | `guess: Trajectory \| PlanningSolution`, `params=None` | `array (n_z,)` | Pack a `Trajectory` or `PlanningSolution` into `z` form |

In [ ]:
x_tilt = np.array([np.pi - 0.5, 0.0])
task = PlanningProblem(plant, x0=x_tilt, cost=cost, U=plant.input_box, horizon=4.0)

short_planner = TrajectoryOptimizationPlanner(
    task.replace(horizon=1.0),
    n_steps=10,
    transcription="single_shooting",
    backend="jax",
    method="adam",
    n_iterations=30,
    learning_rate=0.2,
)
pv = short_planner.compile()
pv.n_z, pv.knot_spacing, pv.predicts, list(pv.params)

In [ ]:
z0 = pv.initial()
z, status = pv.solve(z0=z0)
u_now = pv.command(z, x_tilt, t=0.0)

# Predict state 0.1 s ahead, shift plan to t = 0.1 s, and warm-start the next solve
x_pred = pv.predict_start(x_tilt, u_now, t=0.0, delta=0.1)
z_warm = pv.shift(z, t=0.1)
tick = pv.params  # the face's tree, a copy: the next tick's start and time set in it
tick["x0"], tick["t0"] = x_pred, 0.1
z_next, status_next = pv.solve(params=tick, z0=z_warm)
status, u_now, x_pred, status_next

In [ ]:
# Reconstruct a full PlanningSolution on the host from the packed vector z
reconstructed = pv.solution(z, status=status)
reconstructed.trajectory.x.shape, reconstructed.record.cost

## 4. `ModelPredictiveController`: receding-horizon control as a `System` block

`ModelPredictiveController(planner, *, inputs=None, delay=False, warm_start=True)` wraps `planner.compile()` into a feedback `System` block whose internal state is the last plan vector `z`: `warm_start=True` starts each solve from that plan shifted by one tick, `delay=True` applies at tick $k$ the plan solved at tick $k - 1$, and `inputs=` binds plan numbers to input ports (`{"x0": "y"}` by default). Placed on a clock (`mpc % dt`), it composes with `@` and compiles inside `problem.compile(mpc % dt)`:

In [ ]:
from minilink.control import ModelPredictiveController

mpc = ModelPredictiveController(short_planner)
pe = task.compile(mpc % 0.1, substeps=1, backend="jax")
print(pe.evaluate())

mpc_traj = pe.trajectory()

In [ ]:
pe.loop.plot_trajectory(mpc_traj)

## Summary: the core object pipeline

1. **`System` & `Trajectory`** ([`01_systems`](01_systems.ipynb)): continuous (`DynamicSystem`), composite (`DiagramSystem`), discrete (`StepSystem`), and clocked (`SampledSystem`, `SampledDiagramSystem`) blocks and their `Trajectory` records.
2. **`.params`** ([`02_parameters`](02_parameters.ipynb)): uniform dictionary/tree parameter protocol across every object, functional overrides, and `realize(key, n)`.
3. **`Set`, `InputSet`, `Field`, `Distribution`** ([`03_sets_fields_distributions`](03_sets_fields_distributions.ipynb)): feasibility margins (`margin`, `contains`), scalar fields (`value`, `gradient`, `as_set`, `as_cost`), and probability laws (`sample`, `mean`, `cov`, `support`).
4. **`CostFunction` & `StepCost`** ([`04_costs`](04_costs.ipynb)): running/terminal costs (`g`, `h`, `integrate`, `g_max`, `h_max`), cost algebra (`SumCost`, `ScaledCost`), and sampled stage costs (`SampledCost`).
5. **`PlanningProblem` & `StepFace`** ([`05_problem`](05_problem.ipynb)): task specification, `replace`, stochastic realizations, `discretize`, and `StepFace` (`transition`, `run`).
6. **`ProblemEvaluator`, `Evaluation`, `CompiledEvaluator`** ([`06_compile_and_evaluate`](06_compile_and_evaluate.ipynb)): `problem.compile(controller)`, differentiable `Evaluation` scores, Monte Carlo batches, `compare`, and `sys.compile()`.
7. **`MathematicalProgram` & `Optimizer`** ([`07_program`](07_program.ipynb)): `pe.program(wrt)`, `pack`/`unpack`, general NLPs, parametric solves, and differentiable fixed-step optimizers.
8. **`Planner`, `PlanningSolution`, `CompiledTrajectoryPlanner`, `ModelPredictiveController`** ([`08_planners`](08_planners.ipynb)): offline planning, value functions, side-by-side comparisons, pure compiled planner primitives, and receding-horizon MPC.